In [ ]:
%pip install pandas numpy seaborn matplotlib scikit-learn openpyxl --q

# Análise de Pareto — Global e por Paradigma (por classe)

Lê todos os `Modelos<Paradigma>_CV_Consolidado.xlsx` e, para cada curva (cada aba =
cada ranking), marca o ponto de "joelho" (Kneedle) e o ponto 1-SE (Estatístico). Gera
gráficos globais e por paradigma, a tabela de qualidade, o teste de Mann-Whitney
Proposed vs. Literature, a tabela de custo computacional/escalabilidade e a tabela da
fronteira de Pareto, com exportação para LaTeX em todas.

**Organização por classe**: os notebooks 2 e 4 organizam tudo por classe, em subpastas `Classe_<rótulo>/`. Este notebook detecta essas subpastas e repete a análise uma vez para cada classe, lendo e salvando tudo dentro da própria subpasta. Se não encontrar nenhuma subpasta `Classe_*/`, processa a pasta atual.

**Como usar**: coloque este notebook na mesma pasta raiz onde estão as subpastas
`Classe_*/` (saída do notebook 4) e rode todas as células, em ordem.

### 1. Configurações e descoberta das subpastas de classe

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os
import re
from datetime import datetime
from matplotlib.lines import Line2D

PASTA_RAIZ = os.getcwd()
FILE_PATTERN = "Modelos*_CV_Consolidado.xlsx"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")  # 1 timestamp para todo o lote (todas as classes)

TARGET_SUFFIXES = [
    "_u_Original", "_u_Normalizado",
    "_Ppk_Original", "_Ppk_Normalizado",
    "_Pp_Original", "_Pp_Normalizado",
    "_p_Original", "_p_Normalizado",
    "_np_Original", "_np_Normalizado",
    "_Cpk_Original", "_Cpk_Normalizado",
    "_Cp_Original", "_Cp_Normalizado",
    "_c_Original", "_c_Normalizado",
]


def is_target_file(nome_ranking, suffixes=TARGET_SUFFIXES):
    return any(nome_ranking.endswith(suf) for suf in suffixes)


def extrair_paradigma(filename):
    """Modelos<Paradigma>_CV_Consolidado.xlsx -> <Paradigma> (ex.: LogReg_Lasso)."""
    m = re.match(r"^Modelos(.+?)_CV_", filename)
    return m.group(1) if m else "Desconhecido"


def descobrir_escopos(pasta_raiz):
    """Descobre as subpastas Classe_<rótulo>/ dentro de `pasta_raiz`. Se nenhuma for
    encontrada, devolve [(pasta_raiz, None)] -- um único escopo, a própria pasta atual
    (compatibilidade com o uso de uma única base/classe)."""
    subpastas = sorted(
        d for d in os.listdir(pasta_raiz)
        if d.startswith("Classe_") and os.path.isdir(os.path.join(pasta_raiz, d))
    )
    if not subpastas:
        return [(pasta_raiz, None)]
    return [(os.path.join(pasta_raiz, d), d[len("Classe_"):]) for d in subpastas]


ESCOPOS = descobrir_escopos(PASTA_RAIZ)
if ESCOPOS[0][1] is None:
    print(f"Nenhuma subpasta \'Classe_*\' encontrada -- analisando a pasta atual como um único cenário.")
else:
    print(f"{len(ESCOPOS)} subpasta(s) de classe encontrada(s): {[c for _, c in ESCOPOS]}")


### 2. Funções de cálculo (Kneedle, 1-SE, Fronteira de Pareto) — inalteradas

In [ ]:
def get_kneedle_idx(df, col_x, col_y):
    """Algoritmo 1: Kneedle (Geométrico)"""
    df_sorted = df.sort_values(by=col_x, ascending=True).copy()
    x = df_sorted[col_x].values
    y = df_sorted[col_y].values
    if (x.max() - x.min()) == 0 or (y.max() - y.min()) == 0:
        return df_sorted.index[0]
    x_norm = (x - x.min()) / (x.max() - x.min())
    y_norm = (y - y.min()) / (y.max() - y.min())
    coords = np.vstack((x_norm, y_norm)).T
    vec_line = coords[-1] - coords[0]
    vec_points = coords - coords[0]
    norm = np.linalg.norm(vec_line)
    if norm == 0:
        return df_sorted.index[0]
    distances = (vec_line[0] * vec_points[:, 1] - vec_line[1] * vec_points[:, 0]) / norm
    knee_idx_sorted = np.argmax(np.abs(distances))
    return df_sorted.iloc[knee_idx_sorted].name


def get_statistical_idx(df, col_target, col_std):
    """Algoritmo 2: Estatístico (1-SE Rule)"""
    df_sorted = df.sort_values(by="Nº de Atributos", ascending=True)
    best_idx = df_sorted[col_target].idxmax()
    std_val = df_sorted.loc[best_idx, col_std] if col_std in df_sorted.columns else 0
    limit = df_sorted.loc[best_idx, col_target] - std_val
    candidates = df_sorted[df_sorted[col_target] >= limit]
    return candidates.iloc[0].name if not candidates.empty else best_idx


def get_pareto_frontier(df_subset):
    df_sorted = df_subset.sort_values(by="Nº de Atributos", ascending=True)
    pareto_points = []
    max_acc_so_far = -np.inf
    for _, row in df_sorted.iterrows():
        acc = row["Acurácia Balanceada Média"]
        if acc > max_acc_so_far:
            pareto_points.append(row)
            max_acc_so_far = acc
    return pd.DataFrame(pareto_points)


### 3. Plotagem (reutilizável para o gráfico global e para cada paradigma)

In [ ]:
def plot_analysis_colored(all_points_df, highlights_df, titulo, nome_arquivo_base, pasta_saida):
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(18, 11))

    sns.scatterplot(
        data=all_points_df, x="Nº de Atributos", y="Acurácia Balanceada Média",
        color="lightgray", alpha=0.25, s=30, ax=ax, label="_nolegend_",
    )

    pareto = get_pareto_frontier(all_points_df)
    ax.plot(pareto["Nº de Atributos"], pareto["Acurácia Balanceada Média"],
            color="red", linestyle="--", linewidth=2.2, label="Fronteira de Pareto", zorder=5)

    pareto_tuples = list(zip(pareto["Nº de Atributos"].values, pareto["Acurácia Balanceada Média"].values))
    unique_rankings = highlights_df["Ranking"].unique()

    for ranking in unique_rankings:
        subset = highlights_df[highlights_df["Ranking"] == ranking]
        is_target = is_target_file(ranking)
        color = "tab:blue" if is_target else "tab:green"
        marker_kneedle, marker_stat = "o", "s"
        k_s = 220 if is_target else 120
        s_s = 140 if is_target else 90
        z_k = 12 if is_target else 10
        z_s = 11 if is_target else 9
        clean_name = ranking.replace(".xlsx", "").replace("Modelos", "").replace("_", " ").strip()
        label_for_legend = f"[COMPARAÇÃO] {clean_name}" if is_target else clean_name

        kneedle_pt = subset[subset["Tipo"] == "Kneedle"]
        if not kneedle_pt.empty:
            ax.scatter(kneedle_pt["Nº de Atributos"], kneedle_pt["Acurácia Balanceada Média"],
                       marker=marker_kneedle, s=k_s, color=color, edgecolors="black", linewidths=0.9, zorder=z_k)
            for _, row in kneedle_pt.iterrows():
                if any(np.isclose(row["Nº de Atributos"], px, atol=1e-6, rtol=1e-6) and
                       np.isclose(row["Acurácia Balanceada Média"], py, atol=1e-6, rtol=1e-6)
                       for px, py in pareto_tuples):
                    ax.annotate(label_for_legend, xy=(row["Nº de Atributos"], row["Acurácia Balanceada Média"]),
                                xytext=(6, 6), textcoords="offset points", fontsize=13, fontweight="bold",
                                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.9),
                                zorder=30, ha="left", va="bottom")

        stat_pt = subset[subset["Tipo"] == "Estatístico"]
        if not stat_pt.empty:
            ax.scatter(stat_pt["Nº de Atributos"], stat_pt["Acurácia Balanceada Média"],
                       marker=marker_stat, s=s_s, color=color, edgecolors="black", linewidths=0.9, zorder=z_s)
            for _, row in stat_pt.iterrows():
                if any(np.isclose(row["Nº de Atributos"], px, atol=1e-6, rtol=1e-6) and
                       np.isclose(row["Acurácia Balanceada Média"], py, atol=1e-6, rtol=1e-6)
                       for px, py in pareto_tuples):
                    ax.annotate(label_for_legend, xy=(row["Nº de Atributos"], row["Acurácia Balanceada Média"]),
                                xytext=(6, -8), textcoords="offset points", fontsize=13, fontweight="bold",
                                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.9),
                                zorder=30, ha="left", va="top")

    legend_elements_shapes = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=12,
               markeredgecolor="black", label="Kneedle (bolinha)"),
        Line2D([0], [0], marker="s", color="w", markerfacecolor="gray", markersize=10,
               markeredgecolor="black", label="Estatístico (quadrado)"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue", markersize=10,
               markeredgecolor="black", label="Grupo Comparação (azul)"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:green", markersize=10,
               markeredgecolor="black", label="Outros Arquivos (verde)"),
        Line2D([0], [0], color="red", linestyle="--", lw=2, label="Fronteira de Pareto"),
    ]
    ax.legend(handles=legend_elements_shapes, title="Legenda", loc="upper left",
              bbox_to_anchor=(1.02, 1.0), frameon=True, fontsize=10, title_fontsize=11)

    plt.title(titulo, fontsize=18)
    plt.xlabel("Complexidade (Nº de Atributos)", fontsize=14)
    plt.ylabel("Acurácia Balanceada Média", fontsize=14)
    plt.subplots_adjust(right=0.72)

    # Salva dentro de `pasta_saida` (a subpasta da classe sendo processada), em vez de no diretório de trabalho do Python (que não muda entre classes) -- no notebook original, FOLDER_PATH == cwd sempre, então esse caminho relativo não era um problema; agora com várias subpastas, precisa ser explícito.
    filename = os.path.join(pasta_saida, f"{nome_arquivo_base}_{TIMESTAMP}.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Gráfico gerado: {filename}")


### 4. Leitura dos arquivos consolidados de um escopo

In [ ]:
COLUNAS_EXTRA = ["Sensibilidade Média", "Especificidade Média", "AUC Média"]
METRICAS_SIGNIFICANCIA = ["Acurácia Balanceada", "Sensibilidade", "Especificidade", "AUC"]


def ler_curvas(folder_path):
    full_path = os.path.join(folder_path, FILE_PATTERN)
    todos_os_arquivos = glob.glob(full_path)
    arquivos = [f for f in todos_os_arquivos if "Resumo_Geral" not in os.path.basename(f)]
    excluidos = len(todos_os_arquivos) - len(arquivos)
    print(f"Lendo {len(arquivos)} arquivo(s) consolidado(s) ({excluidos} \'Resumo_Geral\' excluído(s)).")

    all_data_list, highlights_list = [], []

    for file_path in arquivos:
        file_name = os.path.basename(file_path)
        paradigma = extrair_paradigma(file_name)
        try:
            todas_as_abas = pd.read_excel(file_path, sheet_name=None)
        except Exception as e:
            print(f"Erro ao abrir {file_name}: {e}")
            continue

        for nome_ranking, df in todas_as_abas.items():
            try:
                if "Nº de Atributos" not in df.columns or "Acurácia Balanceada Média" not in df.columns:
                    print(f"Aviso: aba \'{nome_ranking}\' de \'{file_name}\' sem colunas necessárias. Pulando.")
                    continue
                colunas_extra_presentes = [c for c in COLUNAS_EXTRA if c in df.columns]
                df_temp = df[["Nº de Atributos", "Acurácia Balanceada Média"]].copy()
                df_temp["Ranking"] = nome_ranking
                df_temp["Paradigma"] = paradigma
                all_data_list.append(df_temp)

                for idx, tipo in [
                    (get_kneedle_idx(df, "Nº de Atributos", "Acurácia Balanceada Média"), "Kneedle"),
                    (get_statistical_idx(df, "Acurácia Balanceada Média", "Acurácia Balanceada Desvio"), "Estatístico"),
                ]:
                    registro = {
                        "Ranking": nome_ranking, "Paradigma": paradigma, "Tipo": tipo,
                        "Nº de Atributos": df.loc[idx, "Nº de Atributos"],
                        "Acurácia Balanceada Média": df.loc[idx, "Acurácia Balanceada Média"],
                    }
                    for col in colunas_extra_presentes:
                        registro[col] = df.loc[idx, col]
                    for metrica in METRICAS_SIGNIFICANCIA:
                        col_fold = f"{metrica}_Por_Fold"
                        if col_fold in df.columns:
                            registro[col_fold] = df.loc[idx, col_fold]
                    highlights_list.append(registro)
            except Exception as e:
                print(f"Erro na aba \'{nome_ranking}\' de {file_name}: {e}")

    if not all_data_list or not highlights_list:
        return None, None
    return pd.concat(all_data_list, ignore_index=True), pd.DataFrame(highlights_list)


### 5. Análise completa de um escopo

A função `processar_escopo` reúne a análise de um escopo: gráficos global e por paradigma, tabela de qualidade com LaTeX, teste de Mann-Whitney Proposed vs. Literature, tabela de custo e escalabilidade com LaTeX e tabela da fronteira de Pareto com LaTeX. `FOLDER_PATH` aponta para a subpasta da classe em processamento.

In [ ]:
def processar_escopo(FOLDER_PATH, classe_rotulo):
    rotulo_exibicao = f"Classe {classe_rotulo}" if classe_rotulo else "(pasta única)"
    print(f"\n{'=' * 70}\n {rotulo_exibicao}  ->  {FOLDER_PATH}\n{'=' * 70}")

    # --- 5. Processamento: gráfico global + um gráfico por paradigma ---
    df_all, df_high = ler_curvas(FOLDER_PATH)
    if df_all is None:
        print(f"Nenhum dado válido encontrado em {rotulo_exibicao}. Pulando este escopo.")
        return

    plot_analysis_colored(
        df_all, df_high,
        titulo=f"{rotulo_exibicao} -- Visão Global: Fronteira de Pareto e Destaques",
        nome_arquivo_base="Analise_Global_Pareto_Destaque", pasta_saida=FOLDER_PATH,
    )

    paradigmas = sorted(df_all["Paradigma"].unique())
    print(f"\nGerando gráficos individuais para {len(paradigmas)} paradigma(s): {paradigmas}")
    for paradigma in paradigmas:
        df_all_p = df_all[df_all["Paradigma"] == paradigma]
        df_high_p = df_high[df_high["Paradigma"] == paradigma]
        if df_all_p.empty or df_high_p.empty:
            continue
        plot_analysis_colored(
            df_all_p, df_high_p,
            titulo=f"{rotulo_exibicao} -- Paradigma: {paradigma} -- Fronteira de Pareto e Destaques",
            nome_arquivo_base=f"Analise_Paradigma_{paradigma}_Pareto_Destaque", pasta_saida=FOLDER_PATH,
        )

    resumo_filename = os.path.join(FOLDER_PATH, f"Resumo_Global_Dados_{TIMESTAMP}.xlsx")
    df_high.to_excel(resumo_filename, index=False)
    print(f"\nResumo salvo: {resumo_filename}")

    # --- 6. Tabela consolidada de qualidade ---
    METRICAS_TABELA = [
        ("Feature Qty. (Mean)", "Nº de Atributos", False, 1),
        ("Sensitivity (%)", "Sensibilidade Média", True, 2),
        ("Specificity (%)", "Especificidade Média", True, 2),
        ("AUC-ROC (%)", "AUC Média", True, 2),
    ]
    ORDEM_PARADIGMAS = ["DT", "RF", "GB", "SVM", "MLP", "GNB", "LogReg_Lasso"]
    ORDEM_APPROACH_TIPO = [
        ("Literature", "Estatístico", "Literature + Statistical"),
        ("Literature", "Kneedle", "Literature + Kneedle"),
        ("Proposed", "Estatístico", "Proposed + Statistical"),
        ("Proposed", "Kneedle", "Proposed + Kneedle"),
    ]

    df_tab = df_high.copy()
    df_tab["Approach"] = df_tab["Ranking"].apply(lambda nome: "Proposed" if is_target_file(nome) else "Literature")

    paradigmas_presentes = [p for p in ORDEM_PARADIGMAS if p in df_tab["Paradigma"].unique()]
    extras = sorted(p for p in df_tab["Paradigma"].unique() if p not in ORDEM_PARADIGMAS)
    if extras:
        print(f"Paradigma(s) fora de ORDEM_PARADIGMAS, acrescentado(s) ao final: {extras}")
        paradigmas_presentes += extras

    valores = {}
    for metric_label, col_fonte, is_percent, casas in METRICAS_TABELA:
        for approach, tipo, approach_tipo_label in ORDEM_APPROACH_TIPO:
            linha = {}
            for paradigma in paradigmas_presentes:
                grupo = df_tab[(df_tab["Paradigma"] == paradigma) & (df_tab["Approach"] == approach) & (df_tab["Tipo"] == tipo)]
                if col_fonte not in df_tab.columns or grupo[col_fonte].dropna().empty:
                    linha[paradigma] = np.nan
                    continue
                media = grupo[col_fonte].mean()
                linha[paradigma] = media * 100 if is_percent else media
            valores[(metric_label, approach_tipo_label)] = linha

    linhas_flat = []
    for (metric_label, approach_tipo_label), linha in valores.items():
        registro = {"Metric": metric_label, "Approach": approach_tipo_label}
        registro.update(linha)
        linhas_flat.append(registro)
    df_tabela_consolidada = pd.DataFrame(linhas_flat)
    caminho_consolidada = os.path.join(FOLDER_PATH, f"Tabela_Consolidada_LiteratureVsProposed_{TIMESTAMP}.xlsx")
    df_tabela_consolidada.to_excel(caminho_consolidada, index=False)
    print(f"Tabela consolidada (Excel) salva em: {caminho_consolidada}")

    negrito = {}
    for metric_label, col_fonte, is_percent, casas in METRICAS_TABELA:
        menor_e_melhor = (metric_label == "Feature Qty. (Mean)")
        for tipo_label_sufixo, lit_label, prop_label in [
            ("Statistical", "Literature + Statistical", "Proposed + Statistical"),
            ("Kneedle", "Literature + Kneedle", "Proposed + Kneedle"),
        ]:
            for paradigma in paradigmas_presentes:
                v_lit = valores[(metric_label, lit_label)].get(paradigma, np.nan)
                v_prop = valores[(metric_label, prop_label)].get(paradigma, np.nan)
                if pd.isna(v_lit) or pd.isna(v_prop):
                    continue
                if menor_e_melhor:
                    vencedor = lit_label if v_lit < v_prop else prop_label
                else:
                    vencedor = lit_label if v_lit > v_prop else prop_label
                negrito[(metric_label, vencedor, paradigma)] = True
    print("Regra de negrito aplicada (Literature vs. Proposed, por estratégia).")

    # --- 7. LaTeX da tabela de qualidade (sem significância) ---
    def fmt(valor, casas, is_bold):
        texto = "--" if pd.isna(valor) else f"{valor:.{casas}f}"
        return f"\\textbf{{{texto}}}" if is_bold else texto

    linhas_latex = ["% Requer \\usepackage{booktabs} e \\usepackage{multirow}",
                     "\\begin{table}[htbp]", "\\centering",
                     "\\caption{Consolidated Quality Evaluation of Classifiers}",
                     "\\label{tab:qualidade_consolidada}",
                     f"\\begin{{tabular}}{{ll{'c' * len(paradigmas_presentes)}}}", "\\toprule",
                     "Metric & Approach & " + " & ".join(paradigmas_presentes) + " \\\\", "\\midrule"]
    for metric_label, col_fonte, is_percent, casas in METRICAS_TABELA:
        metric_label_latex = metric_label.replace("%", "\\%")
        linhas_latex.append(f"\\multirow{{4}}{{*}}{{{metric_label_latex}}}")
        for approach, tipo, approach_tipo_label in ORDEM_APPROACH_TIPO:
            linha = valores[(metric_label, approach_tipo_label)]
            celulas = [fmt(linha.get(p, np.nan), casas, negrito.get((metric_label, approach_tipo_label, p), False))
                       for p in paradigmas_presentes]
            linhas_latex.append(f" & {approach_tipo_label} & " + " & ".join(celulas) + " \\\\")
        linhas_latex.append("\\midrule")
    if linhas_latex[-1] == "\\midrule":
        linhas_latex.pop()
    linhas_latex += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    codigo_latex = "\n".join(linhas_latex)
    caminho_tex = os.path.join(FOLDER_PATH, f"Tabela_Consolidada_LiteratureVsProposed_{TIMESTAMP}.tex")
    with open(caminho_tex, "w", encoding="utf-8") as f:
        f.write(codigo_latex)
    print(f"Código LaTeX salvo em: {caminho_tex}")

    # --- 8. Mann-Whitney: Proposed vs. Literature ---
    from scipy.stats import mannwhitneyu
    METRICAS_MANNWHITNEY = [
        ("Acurácia Balanceada Média", "Acurácia Balanceada"), ("Sensibilidade Média", "Sensibilidade"),
        ("Especificidade Média", "Especificidade"), ("AUC Média", "AUC"),
    ]
    registros_sig = []
    for paradigma in paradigmas_presentes:
        for tipo in ["Kneedle", "Estatístico"]:
            df_p_t = df_tab[(df_tab["Paradigma"] == paradigma) & (df_tab["Tipo"] == tipo)]
            df_prop = df_p_t[df_p_t["Approach"] == "Proposed"]
            df_lit = df_p_t[df_p_t["Approach"] == "Literature"]
            for col_media, rotulo_metrica in METRICAS_MANNWHITNEY:
                if col_media not in df_p_t.columns:
                    continue
                grupo_prop = df_prop[col_media].dropna().values
                grupo_lit = df_lit[col_media].dropna().values
                if len(grupo_prop) < 1 or len(grupo_lit) < 1 or (len(grupo_prop) == 1 and len(grupo_lit) == 1):
                    p_valor, sig = np.nan, False
                else:
                    try:
                        _, p_valor = mannwhitneyu(grupo_prop, grupo_lit, alternative="two-sided")
                        sig = bool(p_valor <= 0.05)
                    except ValueError:
                        p_valor, sig = np.nan, False
                media_prop = float(np.mean(grupo_prop)) if len(grupo_prop) else np.nan
                media_lit = float(np.mean(grupo_lit)) if len(grupo_lit) else np.nan
                registros_sig.append({
                    "Paradigma": paradigma, "Estratégia": tipo, "Métrica": rotulo_metrica,
                    "N Métodos Proposed": len(grupo_prop), "N Métodos Literature": len(grupo_lit),
                    "Média Proposed": round(media_prop, 4) if not pd.isna(media_prop) else np.nan,
                    "Média Literature": round(media_lit, 4) if not pd.isna(media_lit) else np.nan,
                    "p-valor (Mann-Whitney)": round(float(p_valor), 4) if not pd.isna(p_valor) else np.nan,
                    "Diferença Significativa (p<=0.05)": sig,
                    "Quem Venceu": ("Proposed" if (not pd.isna(media_prop) and not pd.isna(media_lit) and media_prop > media_lit)
                                    else "Literature" if not pd.isna(media_lit) else "--"),
                })
    df_significancia = pd.DataFrame(registros_sig)
    caminho_sig = os.path.join(FOLDER_PATH, f"CODE5_Significancia_PropostoVsLiteratura_{TIMESTAMP}.xlsx")
    df_significancia.to_excel(caminho_sig, index=False)
    n_testado = df_significancia["p-valor (Mann-Whitney)"].notna().sum() if not df_significancia.empty else 0
    n_sig = df_significancia["Diferença Significativa (p<=0.05)"].sum() if not df_significancia.empty else 0
    print(f"{n_sig} de {n_testado} comparações com diferença estatisticamente significativa. Salvo: {caminho_sig}")

    # --- 9. Custo computacional / escalabilidade ---
    from scipy.stats import linregress

    def calcular_escalabilidade(df):
        if "Nº de Atributos" not in df.columns or "Tempo de Execução (s)" not in df.columns:
            return None
        df = df[(df["Nº de Atributos"] > 0) & (df["Tempo de Execução (s)"] > 0)]
        if len(df) < 3:
            return None
        log_n = np.log(df["Nº de Atributos"].values)
        log_t = np.log(df["Tempo de Execução (s)"].values)
        slope, intercept, r_value, p_value, std_err = linregress(log_n, log_t)
        return {
            "Scalability (alpha)": slope, "Reliability (r)": r_value,
            "Mean Time (s)": df["Tempo de Execução (s)"].mean(),
            "Efficiency (s/feat.)": (df["Tempo de Execução (s)"] / df["Nº de Atributos"]).mean(),
            "N Pontos Curva": len(df),
        }

    arquivos_consolidados_custo = sorted(glob.glob(os.path.join(FOLDER_PATH, FILE_PATTERN)))
    registros_custo = []
    for arquivo in arquivos_consolidados_custo:
        nome_arquivo = os.path.basename(arquivo)
        paradigma = extrair_paradigma(nome_arquivo)
        try:
            todas_as_abas = pd.read_excel(arquivo, sheet_name=None)
        except Exception as e:
            print(f"Erro ao abrir \'{nome_arquivo}\': {e}")
            continue
        for nome_ranking, df_aba in todas_as_abas.items():
            resultado = calcular_escalabilidade(df_aba)
            if resultado is None:
                continue
            registros_custo.append({"Paradigma": paradigma, "Ranking": nome_ranking,
                                     "Approach": "Proposed" if is_target_file(nome_ranking) else "Literature", **resultado})
    df_custo_bruto = pd.DataFrame(registros_custo)
    caminho_custo_bruto = os.path.join(FOLDER_PATH, f"CODE9_Escalabilidade_Por_Ranking_{TIMESTAMP}.xlsx")
    df_custo_bruto.to_excel(caminho_custo_bruto, index=False)
    print(f"{len(df_custo_bruto)} curva(s) de custo processada(s). Salvo: {caminho_custo_bruto}")

    # --- 9.1 Agregação de custo + significância ---
    METRICAS_CUSTO = [("Mean Time (s)", True), ("Scalability (alpha)", True),
                       ("Reliability (r)", False), ("Efficiency (s/feat.)", True)]
    df_sig_custo = pd.DataFrame()
    valores_custo, negrito_custo, paradigmas_custo = {}, {}, []
    if not df_custo_bruto.empty:
        paradigmas_custo = [p for p in ORDEM_PARADIGMAS if p in df_custo_bruto["Paradigma"].unique()]
        significancia_custo = []
        for col_metrica, menor_e_melhor in METRICAS_CUSTO:
            for approach in ["Literature", "Proposed"]:
                linha = {}
                for paradigma in paradigmas_custo:
                    grupo = df_custo_bruto[(df_custo_bruto["Paradigma"] == paradigma) & (df_custo_bruto["Approach"] == approach)][col_metrica].dropna()
                    linha[paradigma] = grupo.mean() if not grupo.empty else np.nan
                valores_custo[(col_metrica, approach)] = linha
            for paradigma in paradigmas_custo:
                grupo_prop = df_custo_bruto[(df_custo_bruto["Paradigma"] == paradigma) & (df_custo_bruto["Approach"] == "Proposed")][col_metrica].dropna().values
                grupo_lit = df_custo_bruto[(df_custo_bruto["Paradigma"] == paradigma) & (df_custo_bruto["Approach"] == "Literature")][col_metrica].dropna().values
                if len(grupo_prop) < 2 or len(grupo_lit) < 2:
                    p_valor, sig = np.nan, False
                else:
                    try:
                        _, p_valor = mannwhitneyu(grupo_prop, grupo_lit, alternative="two-sided")
                        sig = bool(p_valor <= 0.05)
                    except ValueError:
                        p_valor, sig = np.nan, False
                significancia_custo.append({
                    "Paradigma": paradigma, "Métrica": col_metrica,
                    "N Proposed": len(grupo_prop), "N Literature": len(grupo_lit),
                    "Média Proposed": round(float(np.mean(grupo_prop)), 4) if len(grupo_prop) else np.nan,
                    "Média Literature": round(float(np.mean(grupo_lit)), 4) if len(grupo_lit) else np.nan,
                    "p-valor (Mann-Whitney)": round(float(p_valor), 4) if not pd.isna(p_valor) else np.nan,
                    "Diferença Significativa (p<=0.05)": sig,
                })
        df_sig_custo = pd.DataFrame(significancia_custo)
        caminho_sig_custo = os.path.join(FOLDER_PATH, f"CODE9_Significancia_Custo_{TIMESTAMP}.xlsx")
        df_sig_custo.to_excel(caminho_sig_custo, index=False)
        for col_metrica, menor_e_melhor in METRICAS_CUSTO:
            for paradigma in paradigmas_custo:
                v_lit = valores_custo[(col_metrica, "Literature")].get(paradigma, np.nan)
                v_prop = valores_custo[(col_metrica, "Proposed")].get(paradigma, np.nan)
                if pd.isna(v_lit) or pd.isna(v_prop):
                    continue
                if menor_e_melhor:
                    vencedor = "Literature" if v_lit < v_prop else "Proposed"
                else:
                    vencedor = "Literature" if v_lit > v_prop else "Proposed"
                negrito_custo[(col_metrica, vencedor, paradigma)] = True
        n_sig_c = df_sig_custo["Diferença Significativa (p<=0.05)"].sum()
        print(f"Tabela de custo agregada. {n_sig_c} de {len(df_sig_custo)} comparações significativas. Salvo: {caminho_sig_custo}")
    else:
        print("Sem dados de custo -- seções de custo/escalabilidade puladas.")

    # --- 10. LaTeX: tabela de custo + tabela de qualidade, ambas com † de significância ---
    def fmt_com_significancia(valor, casas, is_bold, is_sig):
        texto = "--" if pd.isna(valor) else f"{valor:.{casas}f}"
        if not pd.isna(valor) and is_sig:
            texto += "$^\\dagger$"
        return f"\\textbf{{{texto}}}" if is_bold else texto

    if not df_sig_custo.empty:
        def sig_custo_lookup(metric_label, approach_label, paradigma):
            linha = df_sig_custo[(df_sig_custo["Métrica"] == metric_label) & (df_sig_custo["Paradigma"] == paradigma)]
            return False if linha.empty else bool(linha.iloc[0]["Diferença Significativa (p<=0.05)"])

        partes_custo = ["% Requer \\usepackage{booktabs}, \\usepackage{multirow}, \\usepackage{amssymb}",
                         "\\begin{table}[htbp]", "\\centering",
                         "\\caption{Consolidated Comparison of Computational Efficiency and Scalability}",
                         "\\label{tab:custo_consolidado}",
                         f"\\begin{{tabular}}{{ll{'c' * len(paradigmas_custo)}}}", "\\toprule",
                         "Metric & Approach & " + " & ".join(paradigmas_custo) + " \\\\", "\\midrule"]
        for col_metrica, menor_e_melhor in METRICAS_CUSTO:
            casas = 2 if col_metrica != "Efficiency (s/feat.)" else 3
            partes_custo.append(f"\\multirow{{2}}{{*}}{{{col_metrica}}}")
            for approach_label, approach_chave in [("Literature (Standard)", "Literature"), ("Proposed (Target)", "Proposed")]:
                celulas = []
                for paradigma in paradigmas_custo:
                    v = valores_custo[(col_metrica, approach_chave)].get(paradigma, np.nan)
                    is_bold = negrito_custo.get((col_metrica, approach_chave, paradigma), False)
                    is_sig = sig_custo_lookup(col_metrica, approach_label, paradigma)
                    celulas.append(fmt_com_significancia(v, casas, is_bold, is_sig))
                partes_custo.append(f" & {approach_label} & " + " & ".join(celulas) + " \\\\")
            partes_custo.append("\\midrule")
        if partes_custo[-1] == "\\midrule":
            partes_custo.pop()
        partes_custo += ["\\bottomrule", "\\end{tabular}", "\\vspace{2pt}\\\\",
                          "{\\footnotesize $^\\dagger$ Diferença estatisticamente significativa "
                          "(Mann-Whitney, $p \\leq 0{,}05$), um ranking = uma observação.}", "\\end{table}"]
        codigo_latex_custo = "\n".join(partes_custo)
        caminho_tex_custo = os.path.join(FOLDER_PATH, f"Tabela_Custo_Computacional_{TIMESTAMP}.tex")
        with open(caminho_tex_custo, "w", encoding="utf-8") as f:
            f.write(codigo_latex_custo)
        print(f"Tabela de custo (LaTeX) salva em: {caminho_tex_custo}")

    # --- 10.1 Tabela de qualidade com † ---
    def sig_qualidade_lookup(metric_label, approach_tipo_label, paradigma):
        if "Kneedle" in approach_tipo_label:
            estrategia = "Kneedle"
        elif "Statistical" in approach_tipo_label:
            estrategia = "Estatístico"
        else:
            return False
        mapa_metrica = {"Sensitivity (%)": "Sensibilidade", "Specificity (%)": "Especificidade",
                        "AUC-ROC (%)": "AUC", "Feature Qty. (Mean)": None}
        metrica_teste = mapa_metrica.get(metric_label)
        if metrica_teste is None:
            return False
        linha = df_significancia[(df_significancia["Paradigma"] == paradigma) &
                                  (df_significancia["Estratégia"] == estrategia) &
                                  (df_significancia["Métrica"] == metrica_teste)]
        return False if linha.empty else bool(linha.iloc[0]["Diferença Significativa (p<=0.05)"])

    partes_qual = ["% Requer \\usepackage{booktabs}, \\usepackage{multirow}, \\usepackage{amssymb}",
                   "\\begin{table}[htbp]", "\\centering",
                   "\\caption{Consolidated Quality Evaluation of Classifiers}",
                   "\\label{tab:qualidade_consolidada_sig}",
                   f"\\begin{{tabular}}{{ll{'c' * len(paradigmas_presentes)}}}", "\\toprule",
                   "Metric & Approach & " + " & ".join(paradigmas_presentes) + " \\\\", "\\midrule"]
    for metric_label, col_fonte, is_percent, casas in METRICAS_TABELA:
        metric_label_latex = metric_label.replace("%", "\\%")
        partes_qual.append(f"\\multirow{{4}}{{*}}{{{metric_label_latex}}}")
        for approach, tipo, approach_tipo_label in ORDEM_APPROACH_TIPO:
            linha = valores[(metric_label, approach_tipo_label)]
            celulas = []
            for paradigma in paradigmas_presentes:
                v = linha.get(paradigma, np.nan)
                is_bold = negrito.get((metric_label, approach_tipo_label, paradigma), False)
                is_sig = sig_qualidade_lookup(metric_label, approach_tipo_label, paradigma)
                celulas.append(fmt_com_significancia(v, casas, is_bold, is_sig))
            partes_qual.append(f" & {approach_tipo_label} & " + " & ".join(celulas) + " \\\\")
        partes_qual.append("\\midrule")
    if partes_qual[-1] == "\\midrule":
        partes_qual.pop()
    partes_qual += ["\\bottomrule", "\\end{tabular}", "\\vspace{2pt}\\\\",
                     "{\\footnotesize $^\\dagger$ Diferença estatisticamente significativa (Mann-Whitney, "
                     "$p \\leq 0{,}05$), um método = uma observação.}", "\\end{table}"]
    codigo_latex_qual = "\n".join(partes_qual)
    caminho_tex_qual = os.path.join(FOLDER_PATH, f"Tabela_Qualidade_ComSignificancia_{TIMESTAMP}.tex")
    with open(caminho_tex_qual, "w", encoding="utf-8") as f:
        f.write(codigo_latex_qual)
    print(f"Tabela de qualidade com significância (LaTeX) salva em: {caminho_tex_qual}")

    # --- 11. Tabela da fronteira de Pareto ---
    def formatar_nome_metrica(nome_ranking):
        nome = nome_ranking.replace(".xlsx", "")
        if "_CV_" in nome:
            nome = nome.split("_CV_", 1)[1]
        nome = nome.replace("Ranking_", "")
        if nome.endswith("_Normalizado"):
            return f"Normalized {nome[:-len('_Normalizado')]}"
        if nome.endswith("_Original"):
            return f"Original {nome[:-len('_Original')]}"
        return nome

    TIPO_EM_INGLES = {"Kneedle": "Kneedle", "Estatístico": "Statistical"}
    registros_fronteira = []
    for paradigma in paradigmas_presentes:
        df_all_p = df_all[df_all["Paradigma"] == paradigma]
        df_tab_p = df_tab[df_tab["Paradigma"] == paradigma]
        if df_all_p.empty or df_tab_p.empty:
            continue
        pareto_p = get_pareto_frontier(df_all_p)
        pareto_tuples_p = list(zip(pareto_p["Nº de Atributos"].values, pareto_p["Acurácia Balanceada Média"].values))
        for _, row in df_tab_p.iterrows():
            esta_na_fronteira = any(
                np.isclose(row["Nº de Atributos"], px, atol=1e-6, rtol=1e-6) and
                np.isclose(row["Acurácia Balanceada Média"], py, atol=1e-6, rtol=1e-6)
                for px, py in pareto_tuples_p)
            if not esta_na_fronteira:
                continue
            registros_fronteira.append({
                "Classifier": paradigma, "Approach": row["Approach"],
                "Optimal Method (Extractor + Cut-off)": f"{formatar_nome_metrica(row['Ranking'])} ({TIPO_EM_INGLES.get(row['Tipo'], row['Tipo'])})",
                "Dim. Cost": int(row["Nº de Atributos"]),
                "Mean Balanced Accuracy": round(float(row["Acurácia Balanceada Média"]), 2),
            })

    df_fronteira = pd.DataFrame(registros_fronteira).drop_duplicates()
    if not df_fronteira.empty:
        ordem_paradigma_map = {p: i for i, p in enumerate(ORDEM_PARADIGMAS)}
        df_fronteira["_op"] = df_fronteira["Classifier"].map(ordem_paradigma_map).fillna(999)
        df_fronteira["_oa"] = df_fronteira["Approach"].map({"Literature": 0, "Proposed": 1})
        df_fronteira = df_fronteira.sort_values(["_op", "_oa", "Dim. Cost"]).drop(columns=["_op", "_oa"]).reset_index(drop=True)

    caminho_fronteira = os.path.join(FOLDER_PATH, f"Tabela_Fronteira_Pareto_{TIMESTAMP}.xlsx")
    df_fronteira.to_excel(caminho_fronteira, index=False)
    print(f"{len(df_fronteira)} ponto(s) sobre a fronteira de Pareto. Salvo: {caminho_fronteira}")

    linhas_tex = ["% Requer \\usepackage{booktabs} e \\usepackage{multirow}",
                  "\\begin{table}[htbp]", "\\centering",
                  "\\caption{Pareto Frontier Synthesis: Optimal Models from the Literature and Proposal}",
                  "\\label{tab:fronteira_pareto}", "\\begin{tabular}{llp{5cm}cc}", "\\toprule",
                  "Classifier & Approach & Optimal Method (Extractor + Cut-off) & Dim. Cost & Mean Balanced Accuracy \\\\",
                  "\\midrule"]
    for classifier in df_fronteira["Classifier"].unique():
        df_c = df_fronteira[df_fronteira["Classifier"] == classifier]
        total_linhas_classifier = len(df_c)
        primeira_linha_classifier = True
        for approach in ["Literature", "Proposed"]:
            df_ca = df_c[df_c["Approach"] == approach]
            if df_ca.empty:
                continue
            n_linhas_approach = len(df_ca)
            for j, (_, row) in enumerate(df_ca.iterrows()):
                prefixo_classifier = f"\\multirow{{{total_linhas_classifier}}}{{*}}{{{classifier}}}" if primeira_linha_classifier else ""
                prefixo_approach = f"\\multirow{{{n_linhas_approach}}}{{*}}{{{approach}}}" if j == 0 else ""
                linhas_tex.append(f"{prefixo_classifier} & {prefixo_approach} & "
                                   f"{row['Optimal Method (Extractor + Cut-off)']} & {row['Dim. Cost']} & "
                                   f"{row['Mean Balanced Accuracy']:.2f} \\\\")
                primeira_linha_classifier = False
        linhas_tex.append("\\midrule")
    if linhas_tex and linhas_tex[-1] == "\\midrule":
        linhas_tex.pop()
    linhas_tex += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    codigo_latex_fronteira = "\n".join(linhas_tex)
    caminho_tex_fronteira = os.path.join(FOLDER_PATH, f"Tabela_Fronteira_Pareto_{TIMESTAMP}.tex")
    with open(caminho_tex_fronteira, "w", encoding="utf-8") as f:
        f.write(codigo_latex_fronteira)
    print(f"Tabela da fronteira (LaTeX) salva em: {caminho_tex_fronteira}")
    print(f"{rotulo_exibicao} concluído.")


### 6. Execução — uma vez por subpasta de classe (ou pasta única)

In [ ]:
for pasta_escopo, classe_rotulo in ESCOPOS:
    processar_escopo(pasta_escopo, classe_rotulo)

print("\nAnálise de Pareto concluída para todos os escopos.")
